In [177]:
from ultralytics import YOLO
import cv2
import numpy as np

In [178]:
model = YOLO('yolo26x-seg.pt')

In [179]:
img_path = '../data/raw/11-2-right.jpg'
img = cv2.imread(img_path)

In [180]:
result = model(img, imgsz=960 , conf=0.10)[0]


0: 736x960 21 cars, 1 motorcycle, 1 truck, 1262.2ms
Speed: 6.0ms preprocess, 1262.2ms inference, 31.0ms postprocess per image at shape (1, 3, 736, 960)


In [181]:
# car, motorcycle, bus, truck
VALID_CLASSES = {2, 3, 5, 7}
CLASS_NAMES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

vehicles = []

# using segmentation masks
if result.masks is not None:

    for i, (box, mask) in enumerate(zip(result.boxes, result.masks.data)):
        cls = int(box.cls[0])
        if cls not in VALID_CLASSES:
            continue

        # resize masks
        mask_np = mask.cpu().numpy()
        mask_resized = cv2.resize(mask_np, (img.shape[1], img.shape[0]))
        mask_binary = (mask_resized > 0.5).astype(np.uint8)

        # find contours
        contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        # obtain bounding box
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)

        vehicles.append({
            'x1': x, 'y1': y, 'x2': x + w, 'y2': y + h,
            'width': w, 'height': h, 'cls': cls,
            'name': CLASS_NAMES[cls]
        })
else:
    print("No masks , using bounding boxes")
    for box in result.boxes:
        cls = int(box.cls[0])
        if cls in VALID_CLASSES:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            vehicles.append({
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'width': x2 - x1, 'height': y2 - y1, 'cls': cls,
                'name': CLASS_NAMES[cls]
            })

print(f"Detected cars: {len(vehicles)}")

Detected cars: 23


In [182]:
# order from left
vehicles.sort(key=lambda v: v['x1'])

# Detected vehicles
for i, v in enumerate(vehicles):
    print(f"{i+1}. {v['name']}: Width={v['width']}px")

1. car: Width=440px
2. car: Width=542px
3. car: Width=407px
4. car: Width=305px
5. motorcycle: Width=169px
6. car: Width=220px
7. car: Width=186px
8. car: Width=220px
9. car: Width=220px
10. car: Width=152px
11. car: Width=339px
12. car: Width=135px
13. car: Width=254px
14. car: Width=254px
15. car: Width=220px
16. car: Width=237px
17. car: Width=203px
18. car: Width=169px
19. car: Width=186px
20. car: Width=169px
21. truck: Width=304px
22. car: Width=152px
23. car: Width=152px


In [183]:
# Gaps
parking_spaces = []

for i in range(len(vehicles) - 1):
    v_prev = vehicles[i]
    v_next = vehicles[i + 1]

    # Horizontal distance
    gap_x = v_next['x1'] - v_prev['x2']

    # Vertical distance
    center_y_prev = (v_prev['y1'] + v_prev['y2']) / 2
    center_y_next = (v_next['y1'] + v_next['y2']) / 2
    gap_y = abs(center_y_next - center_y_prev)

    # Both veicles avg width
    avg_length = (v_prev['width'] + v_next['width']) / 2


    ratio = gap_x / avg_length if avg_length > 0 else 0

    min_ratio = 0.10

    # También verificar alineación vertical
    max_vertical_offset = max(v_prev['height'], v_next['height']) * 1

    is_parking = (min_ratio <= ratio) #and gap_y < max_vertical_offset)

    # Debug info
    print(f"\nPair {i}-{i+1}:")
    print(f"  Gap: {gap_x}px")
    print(f"  Average length: {avg_length:.0f}px")
    print(f"  Ratio: {ratio:.2f}x")
    print(f"  PARKING" if is_parking else f"NO parking")

    if is_parking:
        parking_spaces.append({
            'x1': v_prev['x2'],
            'y1': min(v_prev['y1'], v_next['y1']),
            'x2': v_next['x1'],
            'y2': max(v_prev['y2'], v_next['y2']),
            'gap': gap_x,
            'ratio': ratio,
            'between': (i, i+1)
        })


Pair 0-1:
  Gap: 84px
  Average length: 491px
  Ratio: 0.17x
  PARKING

Pair 1-2:
  Gap: -203px
  Average length: 474px
  Ratio: -0.43x
NO parking

Pair 2-3:
  Gap: -187px
  Average length: 356px
  Ratio: -0.53x
NO parking

Pair 3-4:
  Gap: -17px
  Average length: 237px
  Ratio: -0.07x
NO parking

Pair 4-5:
  Gap: -84px
  Average length: 194px
  Ratio: -0.43x
NO parking

Pair 5-6:
  Gap: -152px
  Average length: 203px
  Ratio: -0.75x
NO parking

Pair 6-7:
  Gap: -119px
  Average length: 203px
  Ratio: -0.59x
NO parking

Pair 7-8:
  Gap: -186px
  Average length: 220px
  Ratio: -0.85x
NO parking

Pair 8-9:
  Gap: -186px
  Average length: 186px
  Ratio: -1.00x
NO parking

Pair 9-10:
  Gap: -118px
  Average length: 246px
  Ratio: -0.48x
NO parking

Pair 10-11:
  Gap: -322px
  Average length: 237px
  Ratio: -1.36x
NO parking

Pair 11-12:
  Gap: -67px
  Average length: 194px
  Ratio: -0.34x
NO parking

Pair 12-13:
  Gap: -187px
  Average length: 254px
  Ratio: -0.74x
NO parking

Pair 13-14:

In [184]:
# Draw elipse in detected entities
for v in vehicles:
    center_x = (v['x1'] + v['x2']) // 2
    center_y = (v['y1'] + v['y2']) // 2
    width = int(v['width'] * 0.9)
    height = int(v['height'] * 0.9)

    cv2.ellipse(img, (center_x, center_y), (width//2, height//2),
                0, 0, 360, (0, 255, 255), 2)

    # Label
    cv2.putText(img, v['name'], (v['x1'], v['y1'] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

In [185]:
# Draw metrics
for i in range(len(vehicles) - 1):
    v_prev = vehicles[i]
    v_next = vehicles[i + 1]

    gap_x = v_next['x1'] - v_prev['x2']
    avg_length = (v_prev['width'] + v_next['width']) / 2
    ratio = gap_x / avg_length

    mid_x = (v_prev['x2'] + v_next['x1']) // 2
    center_y_prev = (v_prev['y1'] + v_prev['y2']) / 2
    center_y_next = (v_next['y1'] + v_next['y2']) / 2
    mid_y = int((center_y_prev + center_y_next) / 2)

    # Draw gap info
    cv2.putText(img, f"Gap:{int(gap_x)} ({ratio:.2f}x)",
                (mid_x - 60, mid_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 100, 0), 2)

In [186]:
#get image center
for s in parking_spaces:
    s["cx"] = (s["x1"] + s["x2"]) // 2
    s["cy"] = (s["y1"] + s["y2"]) // 2


xs = np.array([s["cx"] for s in parking_spaces])
mid_x = np.median(xs)

left_side = [s for s in parking_spaces if s["cx"] < mid_x]
right_side = [s for s in parking_spaces if s["cx"] >= mid_x]



def sort_along_street(spaces):
    # Si hay 0 o 1 coches, no hay nada que ordenar
    if len(spaces) < 2:
        return

    pts = np.array([[s["cx"], s["cy"]] for s in spaces])

    v = pts[-1] - pts[0]
    norm = np.linalg.norm(v)

    # Si el vector es casi cero, abortamos
    if norm < 1e-6:
        return

    v = v / norm

    for s in spaces:
        p = np.array([s["cx"], s["cy"]])
        s["_proj"] = p.dot(v)

    spaces.sort(key=lambda s: s["_proj"])

print("Left:", len(left_side), "Right:", len(right_side))

sort_along_street(left_side)
sort_along_street(right_side)


def draw_gaps(spaces):
    # Caso especial: solo 1 hueco
    if len(spaces) == 1:
        s = spaces[0]

        center_x = s["cx"]
        center_y = s["cy"]

        # Usar el gap real, no la caja entera
        axis_y = max(5, int(s["gap"] // 2))
        axis_x = max(10, int((s["x2"] - s["x1"]) * 0.5))

        cv2.ellipse(
            img,
            center=(center_x, center_y),
            axes=(axis_x, axis_y),
            angle=0,
            startAngle=0,
            endAngle=360,
            color=(0, 255, 0),
            thickness=3
        )
        return

    # Caso normal: varios huecos
    if len(spaces) < 2:
        return

    for i in range(len(spaces) - 1):
        a = spaces[i]
        b = spaces[i + 1]

        center_x = (a["cx"] + b["cx"]) // 2
        center_y = (a["cy"] + b["cy"]) // 2

        gap = abs(b["_proj"] - a["_proj"]) if "_proj" in a and "_proj" in b else 100
        if gap < 20:
            continue

        axis_y = int(gap // 2)
        axis_x = int((a["x2"] - a["x1"]) * 0.4)

        cv2.ellipse(
            img,
            center=(center_x, center_y),
            axes=(axis_x, axis_y),
            angle=0,
            startAngle=0,
            endAngle=360,
            color=(0, 255, 0),
            thickness=3
        )



left_side.sort(key=lambda s: s["cy"])
right_side.sort(key=lambda s: s["cy"])

draw_gaps(left_side)
draw_gaps(right_side)
print(parking_spaces[0].keys())

Left: 0 Right: 1
dict_keys(['x1', 'y1', 'x2', 'y2', 'gap', 'ratio', 'between', 'cx', 'cy'])


In [187]:
print(len(parking_spaces))
print(parking_spaces)


1
[{'x1': 1702, 'y1': 1814, 'x2': 1786, 'y2': 3023, 'gap': 84, 'ratio': 0.1710794297352342, 'between': (0, 1), 'cx': 1744, 'cy': 2418}]


In [188]:
# Add general info
info_text = [
    f"Vehicles: {len(vehicles)}",
    f"Parkings: {len(parking_spaces)}"
]

y_offset = 30
for text in info_text:
    cv2.putText(img, text, (10, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(img, text, (10, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 1)
    y_offset += 35

In [189]:
cv2.imwrite("../data/processed/test14.jpg", img)


True